In [578]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

In [579]:
RUTA = "/home/jovyan/work/data"

### CARGA Y TIPADO

In [580]:
df = pd.read_csv(RUTA + "/data.csv", encoding="utf-8-sig")
print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria:     {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Dimensiones: 737,691 filas × 18 columnas
Memoria:     804.6 MB


In [581]:
# Determinamos la cantidad de datos duplicados
df.duplicated().sum()

np.int64(0)

### VARIABLES RELEVANTES

In [582]:
COLS_ELIMINAR = [
    "codigo_institucion",  # redundante con codigo_snies_programa
    "institucion",         # alta cardinalidad sin valor analítico
    "dpto_ies",            # redundante con dpto_programa
]

df = df.drop(columns=COLS_ELIMINAR)

### ESTANDARIZACIÓN DE DATOS

#### GENERO

In [583]:
print(df["genero"].value_counts().to_string())

genero
Hombre        135581
Mujer         133378
Masculino     108963
Femenino      108238
HOMBRE         65873
MUJER          64722
MASCULINO      60945
FEMENINO       59889
No binario        87
Trans             15


In [584]:
MAPA_GENERO = {
    # Masculino — todas las variantes
    "HOMBRE":    "Masculino",
    "Hombre":    "Masculino",
    "MASCULINO": "Masculino",
    "Masculino": "Masculino",
    # Femenino — todas las variantes
    "MUJER":     "Femenino",
    "Mujer":     "Femenino",
    "FEMENINO":  "Femenino",
    "Femenino":  "Femenino",
    # Otros — conservar pero estandarizar
    "No binario": "No Binario",
    "Trans":      "Trans",
}

df["genero"] = df["genero"].str.strip().map(MAPA_GENERO)

# Verificar resultado
print(df["genero"].value_counts().to_string())
print(f"\nSin mapear (NaN): {df['genero'].isna().sum()}")

genero
Masculino     371362
Femenino      366227
No Binario        87
Trans             15

Sin mapear (NaN): 0


#### METODOLOGÍA

In [585]:
print(df["metodologia"].value_counts().to_string())

metodologia
Presencial                      377749
PRESENCIAL                      137453
Virtual                          56196
Distancia (tradicional)          47247
DISTANCIA (TRADICIONAL)          39098
A distancia                      36378
Distancia (virtual)              27910
DISTANCIA (VIRTUAL)               8703
VIRTUAL                           5812
Presencial-Virtual                 877
Presencial-Dual                     82
Híbrida (Presencial-Virtual)        77
Dual                                71
Virtual-Dual                        22
Virtual-A distancia                 12
Presencial-A distancia               4


In [586]:
# 5 categorías
MAPA_METODOLOGIA = {
    "Presencial":                   "Presencial",
    "PRESENCIAL":                   "Presencial",
    "Presencial-Dual":              "Presencial",
    "Presencial-Virtual":           "Híbrida",
    "Híbrida (Presencial-Virtual)": "Híbrida",
    "Presencial-A distancia":       "Híbrida",
    "Virtual-A distancia":          "Híbrida",
    "Virtual":                      "Virtual",
    "VIRTUAL":                      "Virtual",
    "Distancia (virtual)":          "Virtual",
    "DISTANCIA (VIRTUAL)":          "Virtual",
    "Distancia (tradicional)":      "Distancia",
    "DISTANCIA (TRADICIONAL)":      "Distancia",
    "A distancia":                  "Distancia",
    "Dual":                         "Dual",
    "Virtual-Dual":                 "Dual",
}

df["metodologia"] = df["metodologia"].str.strip().map(MAPA_METODOLOGIA)

# Verificar resultado
print(df["metodologia"].value_counts().to_string())

metodologia
Presencial    515284
Distancia     122723
Virtual        98621
Híbrida          970
Dual              93


#### SECTOR

In [587]:
print(df["sector"].value_counts().to_string())

sector
OFICIAL    310838
PRIVADA    279305
Oficial     80106
Privado     67442


In [588]:
MAPA_SECTOR = {
    "OFICIAL": "Oficial",
    "Oficial": "Oficial",
    "PRIVADA": "Privado",
    "Privado": "Privado",
}

df["sector"] = df["sector"].str.strip().map(MAPA_SECTOR)

print(df["sector"].value_counts().to_string())

sector
Oficial    390944
Privado    346747


#### NIVEL DEL FORMACION

In [589]:
print(df["nivel_formacion"].value_counts().to_string())

nivel_formacion
Universitario                          125218
Universitaria                          105138
UNIVERSITARIA                           74260
Tecnológica                             64734
Maestría                                62297
Tecnológico                             58769
Especialización universitaria           54944
TECNOLOGICA                             38020
Especialización Universitaria           28183
Formación técnica profesional           20330
ESPECIALIZACION                         19790
TECNOLÓGICA                             16407
Doctorado                               10728
ESPECIALIZACIÓN UNIVERSITARIA            9954
Especialización médico quirúrgica        9340
MAESTRIA                                 8927
MAESTRÍA                                 7154
FORMACION TECNICA PROFESIONAL            6346
Especialización Médico Quirúrgica        4718
FORMACIÓN TÉCNICA PROFESIONAL            4383
DOCTORADO                                2901
Especialización te

In [590]:
MAPA_NIVEL_FORMACION = {
    # Universitario
    "Universitario":                        "Universitario",
    "Universitaria":                        "Universitario",
    "UNIVERSITARIA":                        "Universitario",

    # Tecnológico
    "Tecnológica":                          "Tecnológico",
    "Tecnológico":                          "Tecnológico",
    "TECNOLOGICA":                          "Tecnológico",
    "TECNOLÓGICA":                          "Tecnológico",

    # Maestría
    "Maestría":                             "Maestría",
    "MAESTRIA":                             "Maestría",
    "MAESTRÍA":                             "Maestría",

    # Especialización universitaria
    "Especialización universitaria":        "Especialización Universitaria",
    "Especialización Universitaria":        "Especialización Universitaria",
    "ESPECIALIZACION":                      "Especialización Universitaria",
    "ESPECIALIZACIÓN UNIVERSITARIA":        "Especialización Universitaria",

    # Técnico profesional
    "Formación técnica profesional":        "Técnico Profesional",
    "FORMACION TECNICA PROFESIONAL":        "Técnico Profesional",
    "FORMACIÓN TÉCNICA PROFESIONAL":        "Técnico Profesional",

    # Doctorado
    "Doctorado":                            "Doctorado",
    "DOCTORADO":                            "Doctorado",

    # Especialización médico quirúrgica
    "Especialización médico quirúrgica":    "Esp. Médico Quirúrgica",
    "Especialización Médico Quirúrgica":    "Esp. Médico Quirúrgica",
    "ESPECIALIZACIÓN MÉDICO QUIRÚRGICA":    "Esp. Médico Quirúrgica",

    # Especialización tecnológica
    "Especialización tecnológica":          "Esp. Tecnológica",
    "ESPECIALIZACIÓN TECNOLÓGICA":          "Esp. Tecnológica",
    "Especialización Tecnológica":          "Esp. Tecnológica",

    # Especialización técnico profesional
    "Especialización técnico profesional":  "Esp. Técnico Profesional",
    "Especialización Técnico Profesional":  "Esp. Técnico Profesional",
    "ESPECIALIZACIÓN TÉCNICO PROFESION":    "Esp. Técnico Profesional",
}

df["nivel_formacion"] = df["nivel_formacion"].str.strip().map(MAPA_NIVEL_FORMACION)

print(df["nivel_formacion"].value_counts().to_string())

nivel_formacion
Universitario                    304616
Tecnológico                      177930
Especialización Universitaria    112871
Maestría                          78378
Técnico Profesional               31059
Esp. Médico Quirúrgica            15776
Doctorado                         13629
Esp. Tecnológica                   3302
Esp. Técnico Profesional            130


#### NIVEL ACADEMICO

In [591]:
print(df["nivel_academico"].value_counts().to_string())

nivel_academico
PREGRADO    415931
POSGRADO    174212
Pregrado     97674
Posgrado     49874


In [592]:
MAPA_NIVEL_ACADEMICO = {
    "PREGRADO": "Pregrado",
    "Pregrado": "Pregrado",
    "POSGRADO": "Posgrado",
    "Posgrado": "Posgrado",
}

df["nivel_academico"] = df["nivel_academico"].str.strip().map(MAPA_NIVEL_ACADEMICO)
print(df["nivel_academico"].value_counts().to_string())

nivel_academico
Pregrado    513605
Posgrado    224086


#### CARACTER

In [593]:
df[['caracter']].value_counts()

caracter                                     
Universidad                                      351208
UNIVERSIDAD                                      111470
Institución Universitaria/Escuela Tecnológica    101176
Institución Tecnológica                           74971
INSTITUCION TECNOLOGICA                           25206
INSTITUCION UNIVERSITARIA/ESCUELA TECNOLOGICA     21706
Institución Técnica Profesional                   19270
INSTITUCIÓN UNIVERSITARIA/ESCUELA TECNOLÓGICA     12184
INSTITUCIÓN TECNOLÓGICA                           11658
INSTITUCION TECNICA PROFESIONAL                    4720
INSTITUCIÓN TÉCNICA PROFESIONAL                    4122
Name: count, dtype: int64

In [594]:
MAPA_CARACTER = {
    # Universidad
    "Universidad":                                     "Universidad",
    "UNIVERSIDAD":                                     "Universidad",

    # Institución Universitaria
    "Institución Universitaria/Escuela Tecnológica":   "Institución Universitaria",
    "INSTITUCION UNIVERSITARIA/ESCUELA TECNOLOGICA":   "Institución Universitaria",
    "INSTITUCIÓN UNIVERSITARIA/ESCUELA TECNOLÓGICA":   "Institución Universitaria",

    # Institución Tecnológica
    "Institución Tecnológica":                         "Institución Tecnológica",
    "INSTITUCION TECNOLOGICA":                         "Institución Tecnológica",
    "INSTITUCIÓN TECNOLÓGICA":                         "Institución Tecnológica",

    # Institución Técnica Profesional
    "Institución Técnica Profesional":                 "Institución Técnica Profesional",
    "INSTITUCION TECNICA PROFESIONAL":                 "Institución Técnica Profesional",
    "INSTITUCIÓN TÉCNICA PROFESIONAL":                 "Institución Técnica Profesional",
}

df["caracter"] = df["caracter"].str.strip().map(MAPA_CARACTER)
print(df["caracter"].value_counts().to_string())

caracter
Universidad                        462678
Institución Universitaria          135066
Institución Tecnológica            111835
Institución Técnica Profesional     28112


#### DEPARTAMENTO PROGRAMA

In [595]:
print(df["dpto_programa"].value_counts().to_string())

dpto_programa
ANTIOQUIA                                                   52395
Bogotá, D.C.                                                46282
Antioquia                                                   45875
VALLE DEL CAUCA                                             30512
Bogotá D.C.                                                 29700
BOGOTA D.C                                                  25806
Valle del Cauca                                             23626
BOGOTA D.C.                                                 23373
SANTANDER                                                   22966
Santander                                                   20211
CUNDINAMARCA                                                17722
Atlántico                                                   17542
Cundinamarca                                                17076
Boyacá                                                      16374
BOGOTÁ D.C.                                                 14

In [596]:
MAPA_DPTO_PROGRAMA = {
    # Bogotá
    "Bogotá, D.C.":          "Bogotá D.C.",
    "Bogotá D.C.":           "Bogotá D.C.",
    "BOGOTA D.C":            "Bogotá D.C.",
    "BOGOTA D.C.":           "Bogotá D.C.",
    "BOGOTÁ D.C.":           "Bogotá D.C.",
    "BOGOTÁ D.C":            "Bogotá D.C.",

    # Antioquia
    "Antioquia":             "Antioquia",
    "ANTIOQUIA":             "Antioquia",

    # Valle del Cauca
    "Valle del Cauca":       "Valle del Cauca",
    "Valle Del Cauca":       "Valle del Cauca",
    "VALLE DEL CAUCA":       "Valle del Cauca",

    # Santander
    "Santander":             "Santander",
    "SANTANDER":             "Santander",

    # Cundinamarca
    "Cundinamarca":          "Cundinamarca",
    "CUNDINAMARCA":          "Cundinamarca",

    # Atlántico
    "Atlántico":             "Atlántico",
    "ATLANTICO":             "Atlántico",
    "ATLÁNTICO":             "Atlántico",

    # Boyacá
    "Boyacá":                "Boyacá",
    "BOYACA":                "Boyacá",
    "BOYACÁ":                "Boyacá",

    # Norte de Santander
    "Norte de Santander":    "Norte de Santander",
    "Norte De Santander":    "Norte de Santander",
    "NORTE DE SANTANDER":    "Norte de Santander",

    # Tolima
    "Tolima":                "Tolima",
    "TOLIMA":                "Tolima",

    # Bolívar
    "Bolívar":               "Bolívar",
    "BOLIVAR":               "Bolívar",
    "BOLÍVAR":               "Bolívar",

    # Caldas
    "Caldas":                "Caldas",
    "CALDAS":                "Caldas",

    # Cauca
    "Cauca":                 "Cauca",
    "CAUCA":                 "Cauca",

    # Huila
    "Huila":                 "Huila",
    "HUILA":                 "Huila",

    # Nariño
    "Nariño":                "Nariño",
    "NARIÑO":                "Nariño",
    "NARINIO":               "Nariño",
    "NARINO":                "Nariño",

    # Risaralda
    "Risaralda":             "Risaralda",
    "RISARALDA":             "Risaralda",

    # Meta
    "Meta":                  "Meta",
    "META":                  "Meta",

    # Magdalena
    "Magdalena":             "Magdalena",
    "MAGDALENA":             "Magdalena",

    # Córdoba
    "Córdoba":               "Córdoba",
    "CORDOBA":               "Córdoba",
    "CÓRDOBA":               "Córdoba",

    # Cesar
    "Cesar":                 "Cesar",
    "CESAR":                 "Cesar",

    # Sucre
    "Sucre":                 "Sucre",
    "SUCRE":                 "Sucre",

    # Quindío
    "Quindío":               "Quindío",
    "QUINDIO":               "Quindío",
    "QUINDÍO":               "Quindío",

    # Casanare
    "Casanare":              "Casanare",
    "CASANARE":              "Casanare",

    # La Guajira
    "La Guajira":            "La Guajira",
    "LA GUAJIRA":            "La Guajira",
    "GUAJIRA":               "La Guajira",

    # Caquetá
    "Caquetá":               "Caquetá",
    "CAQUETA":               "Caquetá",
    "CAQUETÁ":               "Caquetá",

    # Putumayo
    "Putumayo":              "Putumayo",
    "PUTUMAYO":              "Putumayo",

    # Chocó
    "Chocó":                 "Chocó",
    "CHOCO":                 "Chocó",
    "CHOCÓ":                 "Chocó",

    # Guaviare
    "Guaviare":              "Guaviare",
    "GUAVIARE":              "Guaviare",

    # Arauca
    "Arauca":                "Arauca",
    "ARAUCA":                "Arauca",

    # Amazonas
    "Amazonas":              "Amazonas",
    "AMAZONAS":              "Amazonas",

    # Vichada
    "Vichada":               "Vichada",
    "VICHADA":               "Vichada",

    # Guainía
    "Guainía":               "Guainía",
    "GUAINIA":               "Guainía",
    "GUAINÍA":               "Guainía",

    # Vaupés
    "Vaupés":                "Vaupés",
    "VAUPES":                "Vaupés",
    "VAUPÉS":                "Vaupés",

    # San Andrés
    "Archipiélago de San Andrés, Providencia y Santa Catalina": "San Andrés",
    "Archipiélago De San Andrés, Providencia Y Santa Catalina": "San Andrés",
    "SAN ANDRES Y PROVIDENCIA":                                 "San Andrés",
    "SAN ANDRÉS Y PROVIDENCIA":                                 "San Andrés",
    "ARCHIPIÉLAGO DE SA":                                       "San Andrés",
    "SAN ANDRES Y PROVI":                                       "San Andrés",
}

df["dpto_programa"] = df["dpto_programa"].str.strip().map(MAPA_DPTO_PROGRAMA)

print(f"Valores únicos: {df['dpto_programa'].nunique()}")
print(df["dpto_programa"].value_counts().to_string())

Valores únicos: 33
dpto_programa
Bogotá D.C.           152722
Antioquia              98270
Valle del Cauca        60004
Santander              43177
Atlántico              35256
Cundinamarca           34798
Boyacá                 33544
Bolívar                23877
Norte de Santander     23364
Caldas                 21170
Tolima                 21084
Cauca                  20310
Huila                  20248
Nariño                 18614
Risaralda              17060
Meta                   14374
Magdalena              14258
Córdoba                14143
Cesar                  11128
Quindío                 9226
Sucre                   8537
La Guajira              7598
Casanare                7217
Caquetá                 6481
Chocó                   5145
Putumayo                4915
Guaviare                2670
Amazonas                1983
Arauca                  1930
Guainía                 1608
Vichada                 1534
San Andrés              1180
Vaupés                   266


#### AREA PROGRAMA

In [597]:
print(df["area_conocimiento"].value_counts().to_string())

area_conocimiento
Economía, administración, contaduría y afines    156107
Ingeniería, arquitectura, urbanismo y afines     132251
Ciencias sociales y humanas                       79125
Ciencias de la educación                          54199
Ciencias de la salud                              43834
ECONOMIA ADMINISTRACION CONTADURIA Y AFINES       38749
INGENIERIA ARQUITECTURA URBANISMO Y AFINES        33241
CIENCIAS SOCIALES Y HUMANAS                       24496
Bellas artes                                      22582
ECONOMÍA, ADMINISTRACIÓN, CONTADURÍA Y AFINES     21972
INGENIERÍA, ARQUITECTURA, URBANISMO Y AFINES      17960
Agronomía, veterinaria y afines                   17899
Sin información                                   15346
Matemáticas y ciencias naturales                  15201
CIENCIAS DE LA SALUD                              14518
CIENCIAS DE LA EDUCACION                          12139
Sin clasificar                                    10081
BELLAS ARTES                  

In [598]:
MAPA_AREA = {
    # Economía
    "Economía, administración, contaduría y afines":   "Economía, Administración, Contaduría Y Afines",
    "ECONOMIA ADMINISTRACION CONTADURIA Y AFINES":     "Economía, Administración, Contaduría Y Afines",
    "ECONOMÍA, ADMINISTRACIÓN, CONTADURÍA Y AFINES":   "Economía, Administración, Contaduría Y Afines",

    # Ingeniería
    "Ingeniería, arquitectura, urbanismo y afines":    "Ingeniería, Arquitectura, Urbanismo Y Afines",
    "INGENIERIA ARQUITECTURA URBANISMO Y AFINES":      "Ingeniería, Arquitectura, Urbanismo Y Afines",
    "INGENIERÍA, ARQUITECTURA, URBANISMO Y AFINES":    "Ingeniería, Arquitectura, Urbanismo Y Afines",

    # Ciencias sociales
    "Ciencias sociales y humanas":                     "Ciencias Sociales Y Humanas",
    "CIENCIAS SOCIALES Y HUMANAS":                     "Ciencias Sociales Y Humanas",

    # Ciencias de la educación
    "Ciencias de la educación":                        "Ciencias De La Educación",
    "CIENCIAS DE LA EDUCACION":                        "Ciencias De La Educación",
    "CIENCIAS DE LA EDUCACIÓN":                        "Ciencias De La Educación",

    # Ciencias de la salud
    "Ciencias de la salud":                            "Ciencias De La Salud",
    "CIENCIAS DE LA SALUD":                            "Ciencias De La Salud",

    # Bellas artes
    "Bellas artes":                                    "Bellas Artes",
    "BELLAS ARTES":                                    "Bellas Artes",

    # Agronomía
    "Agronomía, veterinaria y afines":                 "Agronomía, Veterinaria Y Afines",
    "AGRONOMIA VETERINARIA Y AFINES":                  "Agronomía, Veterinaria Y Afines",
    "AGRONOMÍA, VETERINARIA Y AFINES":                 "Agronomía, Veterinaria Y Afines",

    # Matemáticas
    "Matemáticas y ciencias naturales":                "Matemáticas Y Ciencias Naturales",
    "MATEMATICAS Y CIENCIAS NATURALES":                "Matemáticas Y Ciencias Naturales",
    "MATEMÁTICAS Y CIENCIAS NATURALES":                "Matemáticas Y Ciencias Naturales",

    # Sin información
    "Sin información":                                 "Sin Información",
    "Sin clasificar":                                  "Sin Información",
}

df["area_conocimiento"] = df["area_conocimiento"].str.strip().map(MAPA_AREA)

print(f"Valores únicos: {df['area_conocimiento'].nunique()}")
print(df["area_conocimiento"].value_counts().to_string())

Valores únicos: 9
area_conocimiento
Economía, Administración, Contaduría Y Afines    216828
Ingeniería, Arquitectura, Urbanismo Y Afines     183452
Ciencias Sociales Y Humanas                      103621
Ciencias De La Educación                          73468
Ciencias De La Salud                              58352
Bellas Artes                                      29896
Agronomía, Veterinaria Y Afines                   26215
Sin Información                                   25427
Matemáticas Y Ciencias Naturales                  20432


## EDA GENERAL

### CALIDAD DE DATOS

#### NULOS

In [599]:
# Determinamos la cantidad de datos faltantes
df.isnull().sum()

sector                   0
caracter                 0
codigo_snies_programa    0
programa_academico       0
nivel_academico          0
nivel_formacion          0
metodologia              0
area_conocimiento        0
nucleo_conocimiento      0
dpto_programa            0
mpio_programa            0
genero                   0
anio                     0
semestre                 0
matriculados             0
dtype: int64

#### FALTANTES

In [600]:
df.columns.to_list()

['sector',
 'caracter',
 'codigo_snies_programa',
 'programa_academico',
 'nivel_academico',
 'nivel_formacion',
 'metodologia',
 'area_conocimiento',
 'nucleo_conocimiento',
 'dpto_programa',
 'mpio_programa',
 'genero',
 'anio',
 'semestre',
 'matriculados']

#### OUTLIERS MATRICULADOS

In [601]:
# ── 1.1 Outliers en matriculados ──────────────────────────
Q1  = df["matriculados"].quantile(0.25)
Q3  = df["matriculados"].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

outliers = df[df["matriculados"] > limite_superior]
print(f"Límite superior IQR : {limite_superior:,.0f}")
print(f"Outliers detectados : {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)")
print(f"\nTop 10 valores extremos:")
print(
    df.nlargest(10, "matriculados")
    [["programa_academico", "anio", "semestre",
      "genero", "matriculados"]]
    .to_string(index=False)
)

Límite superior IQR : 145
Outliers detectados : 92,595 (12.6%)

Top 10 valores extremos:
                                            programa_academico  anio  semestre    genero  matriculados
               TECNOLOGÍA EN ANÁLISIS Y DESARROLLO DE SOFTWARE  2024         2 Masculino         38738
               TECNOLOGÍA EN ANÁLISIS Y DESARROLLO DE SOFTWARE  2023         2 Masculino         32684
               TECNOLOGÍA EN ANÁLISIS Y DESARROLLO DE SOFTWARE  2024         1 Masculino         31158
               TECNOLOGÍA EN ANÁLISIS Y DESARROLLO DE SOFTWARE  2023         1 Masculino         22563
TECNOLOG¿A EN AN¿LISIS Y DESARROLLO DE SISTEMAS DE INFORMACI¿N  2021         2 Masculino         20250
                             TECNOLOGÍA EN GESTIÓN DE MERCADOS  2021         2  Femenino         19350
                           ADMINISTRACIÓN EN SALUD OCUPACIONAL  2018         1  Femenino         18464
                           ADMINISTRACIÓN EN SALUD OCUPACIONAL  2017         1  Femenin

#### MATRICULADOS EN CERO

In [602]:
# ── 1.2 Matriculados en cero ──────────────────────────────
ceros = df[df["matriculados"] == 0]
print(f"\nFilas con matriculados = 0: {len(ceros):,} ({len(ceros)/len(df)*100:.1f}%)")
print(f"\nPor año:")
print(ceros.groupby("anio").size().to_string())
print(f"\nPor área:")
print(ceros.groupby("area_conocimiento").size().sort_values(ascending=False).to_string())


Filas con matriculados = 0: 0 (0.0%)

Por año:
Series([], )

Por área:
Series([], )


### DATASETS

#### POWERBI

In [603]:
COLS_POWERBI = [
    "codigo_snies_programa",
    "programa_academico",
    "area_conocimiento",
    "nucleo_conocimiento",
    "nivel_academico",
    "nivel_formacion",
    "metodologia",
    "sector",
    "caracter",
    "dpto_programa",
    "mpio_programa",
    "genero",
    "anio",
    "semestre",
    "matriculados",
]

df_powerbi = df[COLS_POWERBI].copy()

# Exportar
RUTA_POWERBI = RUTA + "/data_powerbi.csv"
df_powerbi.to_csv(RUTA_POWERBI, index=False, encoding="utf-8-sig")

print(f"   df_powerbi.csv generado")
print(f"   Filas    : {df_powerbi.shape[0]:,}")
print(f"   Columnas : {df_powerbi.shape[1]}")
print(f"   Ruta     : {RUTA_POWERBI}")
print(f"\n   Columnas:")
for col in df_powerbi.columns:
    print(f"   - {col}")

   df_powerbi.csv generado
   Filas    : 737,691
   Columnas : 15
   Ruta     : /home/jovyan/work/data/data_powerbi.csv

   Columnas:
   - codigo_snies_programa
   - programa_academico
   - area_conocimiento
   - nucleo_conocimiento
   - nivel_academico
   - nivel_formacion
   - metodologia
   - sector
   - caracter
   - dpto_programa
   - mpio_programa
   - genero
   - anio
   - semestre
   - matriculados


## EDA MODELO ARIMA

### DF

In [604]:
# Seleccionamos solo las columnas necesarias para el modelo
df_arima = df[["codigo_snies_programa", "anio", "semestre", "matriculados"]].copy()

In [605]:
print("  Tipos de datos y valores nulos:")
info = pd.DataFrame({
    "dtype":  df_arima.dtypes,
    "nulos":  df_arima.isnull().sum(),
    "% nulo": (df_arima.isnull().mean() * 100).round(2),
})
print(info.to_string())

  Tipos de datos y valores nulos:
                       dtype  nulos  % nulo
codigo_snies_programa  int64      0     0.0
anio                   int64      0     0.0
semestre               int64      0     0.0
matriculados           int64      0     0.0


In [606]:
df_arima["fecha"] = df_arima["anio"].astype(str) + "-" + df_arima["semestre"].map({1: "1", 2: "7"})

df_arima = (
    df_arima.groupby(["fecha", "codigo_snies_programa"], as_index=False)["matriculados"]
    .sum()
    .sort_values(["codigo_snies_programa", "fecha"])
    .reset_index(drop=True)
)

In [607]:
RUTA_ARIMA = RUTA + "/data_arima.csv"
df_arima.to_csv(RUTA_ARIMA, index=False, encoding="utf-8-sig")
 
print(f"   df_powerbi.csv generado")
print(f"   Filas    : {df_arima.shape[0]:,}")
print(f"   Columnas : {df_arima.shape[1]}")
print(f"   Ruta     : {RUTA_ARIMA}")
print(f"\n   Columnas:")
for col in df_arima.columns:
    print(f"   - {col}")

   df_powerbi.csv generado
   Filas    : 247,608
   Columnas : 3
   Ruta     : /home/jovyan/work/data/data_arima.csv

   Columnas:
   - fecha
   - codigo_snies_programa
   - matriculados
